# Лаба 02. Чат-агент на LangChain с памятью фактов в файле

Тема семестрового проекта — **проверка Go-кода по код-стайлу команды**. Здесь это наставник, который знакомится
с новым разработчиком: по одному задаёт вопросы (имя, уровень, опыт, стек, проект, слабые места, цель),
запоминает **факты** в `memory.json` и после перезапуска продолжает с тех вопросов, на которые ответа ещё нет.
Когда всё известно, объясняет правила команды S1–S6 с учётом уровня и слабых мест.

Как устроено:

- **Чат** — цепочка `ChatPromptTemplate | ChatOpenAI`, обёрнутая в `RunnableWithMessageHistory`.
  История сообщений живёт только в RAM текущего сеанса.
- **Память** — `FileFactMemory(BaseChatMemory)`: `load_memory_variables()` отдаёт известные факты и список
  недостающих, `save_context()` извлекает факты из реплики **отдельным LLM-вызовом со structured output**
  (`with_structured_output(UserFacts, method="json_schema")`) и пишет `memory.json`.
  Файл есть — читаем, нет — пустая память. Ключ `user_id` — несколько профилей в одном файле.
- **Проверка фактов.** 3B-модель заполняет поля «на всякий случай» (имя «junior», проект «Docker»), поэтому
  извлечённый факт записывается, только если он есть в реплике (`accept_fact`), а уже известные факты
  не перезаписываются. Отброшенное видно в служебном логе.
- **Служебная информация** (что прочитано, извлечено, записано, какой вопрос дальше) выводится в отдельную
  ячейку «Служебный лог» через `display_id`, а не в чат.

Модель — Qwen2.5-3B-Instruct Q4_K_M в llama-server (OpenAI-совместимый API). Для `json_schema`
llama-server строит грамматику по схеме, поэтому даже 3B-модель всегда отдаёт валидный JSON.

```bash
pip install -r requirements.txt
llama-server -m qwen2.5-3b-instruct-q4_k_m.gguf -c 8192
```

In [1]:
import os

# OpenAI-совместимый сервер: llama-server (локально или на Kaggle) или OpenRouter
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # OpenRouter: https://openrouter.ai/api/v1
os.environ.setdefault("LLM_MODEL", "local")                         # OpenRouter: имя модели, ключ в LLM_API_KEY

MEMORY_FILE = "memory.json"
USER_ID = "artur"  # ключ профиля в memory.json

Код агента сохраняется в модуль, чтобы после перезапуска ядра его можно было просто импортировать заново.

In [2]:
%%writefile fact_agent.py
"""Чат-наставник по Go-код-стайлу команды с долгосрочной памятью фактов в файле.

Память — не история сообщений, а набор фактов о пользователе (имя, уровень, стек, цели...).
Факты извлекаются отдельным LLM-вызовом со structured output и хранятся в memory.json по user_id.
История сообщений живёт только в RAM текущего сеанса и после перезапуска пропадает.
"""

import json
import os
import re
import warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Literal, Optional

from IPython.display import Markdown, display
from langchain_classic.memory.chat_memory import BaseChatMemory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core._api import LangChainDeprecationWarning
from pydantic import BaseModel, Field, create_model

# предупреждения об устаревших классах мешаются с выводом чата
warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

STYLE_GUIDE = """\
S1. Имена — MixedCaps/camelCase, без подчёркиваний. Аббревиатуры целиком: userID, не userId.
S2. Ошибки не игнорировать: ни `_ = f()`, ни вызов без проверки возвращаемой ошибки. `defer x.Close()` допускается.
S3. В HTTP-обработчиках не использовать panic — вернуть 500 через http.Error.
S4. Функции, которые ходят в БД или сеть, принимают ctx context.Context первым параметром.
    HTTP-обработчики берут контекст из r.Context() — отдельный параметр ctx им не нужен.
S5. Ошибки, возвращаемые наверх, оборачиваются: fmt.Errorf("контекст: %w", err).
S6. У экспортируемых функций и методов есть doc-комментарий, начинающийся с имени."""


class UserFacts(BaseModel):
    """Факты о разработчике. Поле заполняется, только если пользователь сам это сказал."""

    name: Optional[str] = Field(None, description="Имя пользователя")
    # Literal превращается в enum схемы: грамматика llama-server не даст записать сюда «год»
    go_level: Optional[Literal["junior", "middle", "senior"]] = Field(None, description="Уровень в Go")
    go_experience: Optional[str] = Field(None, description="Сколько времени пишет на Go: «1 год», «полгода»")
    stack: Optional[str] = Field(None, description="Технологии кроме Go: базы данных, инструменты, ОС")
    project: Optional[str] = Field(None, description="Над каким проектом или сервисом работает")
    weak_spots: Optional[str] = Field(None, description="Что даётся трудно в Go или в код-стайле")
    goal: Optional[str] = Field(None, description="Чего хочет добиться")


# порядок опроса и дословные вопросы: 3B-модель, получив «узнай уровень», спрашивает что-то своё
QUESTIONS = {
    "name": "Как тебя зовут?",
    "go_level": "Какой у тебя уровень в Go: junior, middle или senior?",
    "go_experience": "Сколько ты уже пишешь на Go?",
    "stack": "С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?",
    "project": "Над каким проектом ты сейчас работаешь?",
    "weak_spots": "Что тебе сейчас даётся труднее всего в Go или в код-стайле?",
    "goal": "Чего ты хочешь добиться?",
}
LABELS = {
    "name": "Имя", "go_level": "Уровень", "go_experience": "Опыт в Go", "stack": "Стек",
    "project": "Проект", "weak_spots": "Слабые места", "goal": "Цель",
}
INCIDENTAL = ("name", "go_level", "go_experience")  # можно назвать, отвечая на другой вопрос
EMPTY = {"", "null", "none", "нет", "неизвестно", "не указано", "n/a"}


def make_llm(temperature: float = 0.3) -> ChatOpenAI:
    """Любой OpenAI-совместимый сервер: llama-server локально/на Kaggle или OpenRouter."""
    return ChatOpenAI(
        base_url=os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1"),
        api_key=os.environ.get("LLM_API_KEY", "none"),
        model=os.environ.get("LLM_MODEL", "local"),
        temperature=temperature,
        max_tokens=400,
    )


class LogCell:
    """Служебный вывод в собственной ячейке ноутбука, а не в чате.

    Ячейка, в которой создан LogCell, держит display_id; все записи обновляют её вывод.
    """

    def __init__(self, title: str = "Служебный лог"):
        self.title = title
        self.lines: list[str] = []
        self.handle = display(Markdown(self._render()), display_id=True)

    def __call__(self, text: str):
        self.lines.append(f"- `{datetime.now():%H:%M:%S}` {text}")
        self.handle.update(Markdown(self._render()))

    def _render(self) -> str:
        return f"**{self.title}**\n\n" + ("\n".join(self.lines) or "_пока пусто_")


def format_facts(facts: dict) -> str:
    if not facts:
        return "пока ничего не известно"
    return "\n".join(f"- {LABELS[k]}: {v}" for k, v in facts.items() if k in LABELS)


EXTRACT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты извлекаешь факты о пользователе из его последнего сообщения и возвращаешь JSON по схеме.\n"
     "Заполняй поле, только если пользователь сам прямо сказал это в своём сообщении. "
     "Всё, о чём он не сказал, — null. Не придумывай, не копируй слова из вопроса ассистента. "
     "Пиши значения коротко, по-русски."),
    ("human", "Ассистент спрашивал про поле «{field}».\nВопрос ассистента: {question}\n"
              "Сообщение пользователя: {message}"),
])


LEVEL_WORDS = {"junior": ("junior", "джун"), "middle": ("middle", "мидл"), "senior": ("senior", "сеньор", "синьор")}


def accept_fact(key: str, value: str, message: str, asked: str) -> bool:
    """Проверка, что факт действительно есть в реплике: маленькая модель заполняет поля «на всякий случай»."""
    text = message.lower()
    if key == "name":
        return value.lower() in text and (asked == "name" or "зовут" in text or re.search(r"\bя\s", text))
    if key == "go_level":
        return any(word in text for word in LEVEL_WORDS[value])
    if key == "go_experience":
        unit = r"\d|год|лет|месяц|полгода|недел"
        return bool(re.search(unit, value.lower())) and bool(re.search(unit, text))
    # стек, проект, слабые места, цель — только в ответ на свой вопрос
    return key == asked


class FileFactMemory(BaseChatMemory):
    """Долгосрочная память фактов в JSON-файле: {user_id: {"facts": {...}, "updated_at": ...}}.

    load_memory_variables() — отдаёт известные факты и список недостающих;
    save_context() — извлекает факты из реплики пользователя отдельным LLM-вызовом и пишет файл.
    """

    path: Path
    user_id: str
    llm: Any = None  # модель для извлечения фактов
    log: Any = None
    facts: dict = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        profile = self._read_all().get(self.user_id)
        self.facts = dict(profile["facts"]) if profile else {}
        if self.log:
            if not self.path.exists():
                self.log(f"`{self.path}` нет — память пустая")
            elif profile:
                self.log(f"прочитан `{self.path}`, профиль `{self.user_id}`: "
                         f"`{json.dumps(self.facts, ensure_ascii=False)}`")
            else:
                self.log(f"в `{self.path}` нет профиля `{self.user_id}` — память пустая")

    @property
    def memory_variables(self) -> list[str]:
        return ["facts", "missing"]

    def missing(self) -> list[str]:
        return [k for k in QUESTIONS if k not in self.facts]

    def load_memory_variables(self, inputs: dict[str, Any]) -> dict[str, Any]:
        return {"facts": format_facts(self.facts), "missing": self.missing()}

    def save_context(self, inputs: dict[str, Any], outputs: dict[str, str]) -> None:
        """inputs: {"input": реплика, "question": вопрос ассистента, "field": о каком факте он был}.

        Историю сообщений не сохраняем — только факты, прошедшие проверку.
        """
        if not self.missing():
            if self.log:
                self.log("все факты известны — извлечение не нужно")
            return
        message, field = inputs["input"], inputs.get("field") or "—"
        # В схеме — поле, о котором спросили, и недостающие короткие факты, которые называют попутно.
        # Иначе 3B-модель раскладывает перечисление по полям: stack="PostgreSQL", project="Docker"...
        fields = [k for k in self.missing() if k == field or k in INCIDENTAL]
        spec = {k: (UserFacts.model_fields[k].annotation, UserFacts.model_fields[k]) for k in fields}
        if field in fields and field not in INCIDENTAL:
            # Необязательное поле модель охотно оставляет null («Биллинг: считаем платежи...» → project=null),
            # поэтому спрошенное поле — обязательная строка с выходом «нет».
            spec[field] = (str, Field(description=UserFacts.model_fields[field].description
                                      + "; если пользователь об этом не сказал — «нет»"))
        schema = create_model("UserFacts", **spec)
        extractor = EXTRACT_PROMPT | self.llm.with_structured_output(schema, method="json_schema")
        result = extractor.invoke({"message": message, "question": inputs.get("question") or "—", "field": field})
        raw = {k: v.strip() for k, v in result.model_dump().items()
               if isinstance(v, str) and v.strip().lower() not in EMPTY}
        accepted, rejected = {}, {}
        for key, value in raw.items():
            if key in self.facts:
                rejected[key] = "уже известно"
            elif not accept_fact(key, value, message, field):
                rejected[key] = value
            else:
                accepted[key] = value
        if self.log:
            self.log(f"модель извлекла: `{json.dumps(raw, ensure_ascii=False)}`")
            if rejected:
                self.log(f"отброшено проверкой: `{json.dumps(rejected, ensure_ascii=False)}`")
        if accepted:
            self.facts.update(accepted)
            self._write()
            if self.log:
                self.log(f"записано в `{self.path}`: {', '.join(accepted)}")

    def clear(self) -> None:
        self.facts = {}
        self._write()

    def _read_all(self) -> dict:
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text(encoding="utf-8") or "{}")

    def _write(self) -> None:
        data = self._read_all()
        data[self.user_id] = {"facts": self.facts, "updated_at": datetime.now().isoformat(timespec="seconds")}
        self.path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


CHAT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Ты — наставник по код-стайлу Go в команде разработки. Отвечай только по-русски, без английских "
     "и китайских слов (кроме кода), на «ты», кратко: 2–4 предложения.\n\n"
     "Что ты знаешь о пользователе (долгосрочная память):\n{facts}\n\n"
     "Задача на этот ход: {task}"),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])


class FactAgent:
    def __init__(self, user_id: str, path: str | Path = "memory.json", log: LogCell | None = None):
        self.log = log or (lambda text: None)
        self.memory = FileFactMemory(
            path=Path(path), user_id=user_id, log=log, llm=make_llm(0))
        self.chain = CHAT_PROMPT | make_llm(0.3)
        self._histories: dict[str, InMemoryChatMessageHistory] = {}
        self.chat_chain = RunnableWithMessageHistory(
            self.chain, self._history, input_messages_key="input", history_messages_key="history")
        self.last_question = ""
        self.asked = ""  # о каком факте был последний вопрос ассистента

    def _history(self, session_id: str) -> InMemoryChatMessageHistory:
        return self._histories.setdefault(session_id, InMemoryChatMessageHistory())

    def _task(self, start: bool) -> str:
        missing = self.memory.missing()
        if missing:
            ask = f"затем задай дословно этот вопрос и больше ничего не спрашивай: «{QUESTIONS[missing[0]]}»"
            if start and not self.memory.facts:
                return "поздоровайся и одной фразой представься; " + ask
            if start:
                return "пользователь вернулся: поприветствуй его по имени и одной фразой напомни, что уже знаешь; " + ask
            return "одной фразой отреагируй на ответ пользователя (если он что-то спросил — коротко ответь); " + ask
        guide = ("помогай с код-стайлом Go по правилам команды, учитывай уровень, проект и слабые места "
                 "пользователя. Если просят пример — сам напиши короткий пример кода на Go (до 10 строк) "
                 f"из его проекта: как плохо и как по правилам. Правила команды:\n{STYLE_GUIDE}")
        if start:
            return "пользователь вернулся, о нём известно всё нужное: поприветствуй по имени и предложи разобрать " \
                   "его слабое место. Дальше " + guide
        return guide

    def start(self) -> str:
        """Первая реплика сеанса — без ввода пользователя."""
        self.log(f"сеанс начат: user_id=`{self.memory.user_id}`, pid процесса `{os.getpid()}`")
        self.log(f"не хватает фактов: {', '.join(self.memory.missing()) or 'нет'}")
        text = self.chain.invoke({"facts": format_facts(self.memory.facts), "task": self._task(start=True),
                                  "history": [], "input": "(пользователь открыл чат)"}).content.strip()
        self._history(self.memory.user_id).add_ai_message(text)
        self.last_question = text
        self.asked = next(iter(self.memory.missing()), "")
        return text

    def turn(self, user_text: str) -> str:
        # сначала факты: следующий вопрос должен учитывать только что сказанное
        self.memory.save_context(
            {"input": user_text, "question": self.last_question, "field": self.asked}, {})
        missing = self.memory.missing()
        self.log(f"дальше: {'спросить ' + missing[0] if missing else 'режим наставника — все факты собраны'}")
        variables = self.memory.load_memory_variables({})
        text = self.chat_chain.invoke(
            {"input": user_text, "facts": variables["facts"], "task": self._task(start=False)},
            config={"configurable": {"session_id": self.memory.user_id}},
        ).content.strip()
        self.last_question = text
        self.asked = next(iter(missing), "")
        return text

    def chat(self, script: list[str] | None = None):
        """REPL. script — заранее заданные реплики (для демонстрации); без него читает input()."""
        print(f"ассистент> {self.start()}\n")
        replies = iter(script) if script is not None else None
        while True:
            if replies is None:
                user_text = input("вы> ").strip()
            else:
                user_text = next(replies, "/exit")
                print(f"вы> {user_text}")
            if user_text in ("/exit", "/quit", "выход"):
                self.log("сеанс завершён командой выхода")
                break
            if user_text:
                print(f"ассистент> {self.turn(user_text)}\n")


Writing fact_agent.py


## Сеанс 1 — первый запуск

Для воспроизводимой демонстрации реплики пользователя заданы списком; `agent.chat()` без аргументов читает ввод через `input()`.

In [3]:
from pathlib import Path

Path(MEMORY_FILE).unlink(missing_ok=True)  # чистый старт — только для демонстрации

import fact_agent as fa

**Служебный лог — отдельная ячейка.** Агент пишет сюда, пока идёт чат в ячейке ниже.

In [4]:
log = fa.LogCell("Служебный лог — сеанс 1")

**Служебный лог — сеанс 1**

- `11:33:46` `memory.json` нет — память пустая
- `11:33:46` сеанс начат: user_id=`artur`, pid процесса `58`
- `11:33:46` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `11:33:47` модель извлекла: `{"name": "Артур"}`
- `11:33:47` записано в `memory.json`: name
- `11:33:47` дальше: спросить go_level
- `11:33:48` модель извлекла: `{"go_level": "junior", "go_experience": "более года"}`
- `11:33:48` записано в `memory.json`: go_level, go_experience
- `11:33:48` дальше: спросить stack
- `11:33:48` сеанс завершён командой выхода

In [5]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log)
agent.chat(script=[
    'Привет! Меня зовут Артур.',
    'Пишу на Go около года, наверное junior.',
    '/exit',
])

ассистент> Привет! Я твой код-стайл-гуру. Как тебя зовут?

вы> Привет! Меня зовут Артур.


ассистент> Какой у тебя уровень в Go: junior, middle или senior?

вы> Пишу на Go около года, наверное junior.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> /exit


Что осталось на диске после сеанса — только факты, без истории сообщений:

In [6]:
print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года"
    },
    "updated_at": "2026-10-23T11:33:48"
  }
}


## Перезапуск ядра

**Kernel → Restart.** Всё, что было в памяти процесса — агент, история сообщений, переменные, — пропало. Ячейки ниже выполнены в новом ядре: номера выполнения начинаются заново, pid процесса в логе другой.

In [1]:
import os

# OpenAI-совместимый сервер: llama-server (локально или на Kaggle) или OpenRouter
os.environ.setdefault("LLM_BASE_URL", "http://127.0.0.1:8080/v1")  # OpenRouter: https://openrouter.ai/api/v1
os.environ.setdefault("LLM_MODEL", "local")                         # OpenRouter: имя модели, ключ в LLM_API_KEY

MEMORY_FILE = "memory.json"
USER_ID = "artur"  # ключ профиля в memory.json

from pathlib import Path

import fact_agent as fa

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года"
    },
    "updated_at": "2026-10-23T11:33:48"
  }
}


## Сеанс 2 — после перезапуска

Агент читает `memory.json`, здоровается по имени и продолжает с первого вопроса без ответа.

In [2]:
log = fa.LogCell("Служебный лог — сеанс 2")

**Служебный лог — сеанс 2**

- `11:33:52` прочитан `memory.json`, профиль `artur`: `{"name": "Артур", "go_level": "junior", "go_experience": "более года"}`
- `11:33:52` сеанс начат: user_id=`artur`, pid процесса `81`
- `11:33:52` не хватает фактов: stack, project, weak_spots, goal
- `11:33:53` модель извлекла: `{"stack": "PostgreSQL, Docker, NixOS"}`
- `11:33:53` записано в `memory.json`: stack
- `11:33:53` дальше: спросить project
- `11:33:53` модель извлекла: `{"project": "сервис заказов с REST API"}`
- `11:33:53` записано в `memory.json`: project
- `11:33:53` дальше: спросить weak_spots
- `11:33:54` модель извлекла: `{"weak_spots": "управление ошибками в Go"}`
- `11:33:54` записано в `memory.json`: weak_spots
- `11:33:54` дальше: спросить goal
- `11:33:54` модель извлекла: `{"goal": "писать идиоматичный Go и проходить ревью с первого раза"}`
- `11:33:54` записано в `memory.json`: goal
- `11:33:54` дальше: режим наставника — все факты собраны
- `11:33:55` все факты известны — извлечение не нужно
- `11:33:55` дальше: режим наставника — все факты собраны
- `11:33:57` сеанс завершён командой выхода

In [3]:
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=log)
agent.chat(script=[
    'Go с PostgreSQL и Docker, сижу на NixOS.',
    'Делаем сервис заказов с REST API.',
    'Путаюсь, когда надо оборачивать ошибки через fmt.Errorf, а когда можно просто вернуть err.',
    'Хочу писать идиоматичный Go и проходить ревью с первого раза.',
    'Давай, объясни на примере из нашего сервиса.',
    '/exit',
])

ассистент> Привет, Артур! Уже знаешь, что ты разработчик Go. С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> Go с PostgreSQL и Docker, сижу на NixOS.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Делаем сервис заказов с REST API.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайл?

вы> Путаюсь, когда надо оборачивать ошибки через fmt.Errorf, а когда можно просто вернуть err.


ассистент> Чего ты хочешь добиться?

вы> Хочу писать идиоматичный Go и проходить ревью с первого раза.


ассистент> Понял, давай пример: функция, которая может вернуть ошибку при работе с БД. Как сейчас, и как по правилам команды.

вы> Давай, объясни на примере из нашего сервиса.


ассистент> Конечно, давай пример функции для работы с базой данных:

**Сейчас:**
```go
func GetUser(id int) (*User, error) {
    user := &User{}
    err := db.Get(user, "SELECT * FROM users WHERE id = ?", id)
    if err != nil {
        return nil, err
    }
    return user, nil
}
```

**По правилам:**
```go
func GetUser(ctx context.Context, id int) (*User, error) {
    user := &User{}
    query := "SELECT * FROM users WHERE id = ?"
    err := db.Get(user, query, id)
    if err != nil {
        return nil, fmt.Errorf("context: %w", err)
    }
    return user, nil
}
```
Здесь мы добавили контекст ctx первым параметром, а также оборачиваем ошибку в формате, указывающем контекст вызова.

вы> /exit


In [4]:
print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-23T11:33:54"
  }
}


## Второй профиль

Другой `user_id` — свой набор фактов в том же файле; профиль `artur` не затрагивается.

In [5]:
log = fa.LogCell("Служебный лог — профиль guest")

**Служебный лог — профиль guest**

- `11:33:57` в `memory.json` нет профиля `guest` — память пустая
- `11:33:57` сеанс начат: user_id=`guest`, pid процесса `81`
- `11:33:57` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `11:33:58` модель извлекла: `{"name": "Мария", "go_level": "senior", "go_experience": "6 лет"}`
- `11:33:58` записано в `memory.json`: name, go_level, go_experience
- `11:33:58` дальше: спросить stack
- `11:33:58` сеанс завершён командой выхода

In [6]:
guest = fa.FactAgent("guest", MEMORY_FILE, log=log)
guest.chat(script=["Привет, я Мария, senior, пишу на Go шесть лет.", "/exit"])

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

ассистент> Привет! Я код-стайлер по Go. Как тебя зовут?

вы> Привет, я Мария, senior, пишу на Go шесть лет.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> /exit


{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-23T11:33:54"
  },
  "guest": {
    "facts": {
      "name": "Мария",
      "go_level": "senior",
      "go_experience": "6 лет"
    },
    "updated_at": "2026-10-23T11:33:58"
  }
}


## Ещё профили

Полное знакомство с другим разработчиком (`dmitry`) до режима наставника; неполный профиль (`olga`) — пользователь ушёл после двух реплик, недостающие факты останутся в `missing`; возвращение `guest` — агент продолжает со стека, не переспрашивая имя, уровень и опыт.

In [7]:
log = fa.LogCell("Служебный лог — профиль dmitry")

**Служебный лог — профиль dmitry**

- `11:33:58` в `memory.json` нет профиля `dmitry` — память пустая
- `11:33:58` сеанс начат: user_id=`dmitry`, pid процесса `81`
- `11:33:58` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `11:33:59` модель извлекла: `{"name": "Дмитрий"}`
- `11:33:59` записано в `memory.json`: name
- `11:33:59` дальше: спросить go_level
- `11:34:00` модель извлекла: `{"go_level": "middle"}`
- `11:34:00` записано в `memory.json`: go_level
- `11:34:00` дальше: спросить go_experience
- `11:34:00` модель извлекла: `{"go_experience": "три года"}`
- `11:34:00` записано в `memory.json`: go_experience
- `11:34:00` дальше: спросить stack
- `11:34:01` модель извлекла: `{"stack": "Kafka, Redis, ClickHouse, Kubernetes"}`
- `11:34:01` записано в `memory.json`: stack
- `11:34:01` дальше: спросить project
- `11:34:01` модель извлекла: `{"project": "Биллинг"}`
- `11:34:01` записано в `memory.json`: project
- `11:34:01` дальше: спросить weak_spots
- `11:34:02` модель извлекла: `{"weak_spots": "context.Context"}`
- `11:34:02` записано в `memory.json`: weak_spots
- `11:34:02` дальше: спросить goal
- `11:34:02` модель извлекла: `{"goal": "сам ревьюить код команды и не пропускать ошибки"}`
- `11:34:02` записано в `memory.json`: goal
- `11:34:02` дальше: режим наставника — все факты собраны
- `11:34:05` все факты известны — извлечение не нужно
- `11:34:05` дальше: режим наставника — все факты собраны
- `11:34:07` сеанс завершён командой выхода

In [8]:
dmitry = fa.FactAgent("dmitry", MEMORY_FILE, log=log)
dmitry.chat(script=[
    'Здравствуйте, меня зовут Дмитрий.',
    'Скорее middle.',
    'Года три, до этого писал на Python.',
    'Kafka, Redis, ClickHouse, всё крутится в Kubernetes.',
    'Биллинг: считаем платежи и выставляем счета клиентам.',
    'Не всегда понимаю, в какие функции нужно передавать context.Context.',
    'Хочу сам ревьюить код команды и не пропускать ошибки.',
    'Покажи, как правильно передать ctx в запрос к ClickHouse.',
    '/exit',
])

ассистент> Привет! Я код-стайлер Go. Как тебя зовут?

вы> Здравствуйте, меня зовут Дмитрий.


ассистент> Какой у тебя уровень в Go: junior, middle или senior?

вы> Скорее middle.


ассистент> Сколько ты уже пишешь на Go?

вы> Года три, до этого писал на Python.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> Kafka, Redis, ClickHouse, всё крутится в Kubernetes.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Биллинг: считаем платежи и выставляем счета клиентам.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайле?

вы> Не всегда понимаю, в какие функции нужно передавать context.Context.


ассистент> Чего ты хочешь добиться?

вы> Хочу сам ревьюить код команды и не пропускать ошибки.


ассистент> Понял, чтобы улучшить качество кода, начни с функций, которые взаимодействуют с базами данных или сетью. Например, вот как правильно и неправильно использовать context.Context:

**Неправильно:**
```go
func getUser(ctx context.Context, userID int) (*User, error) {
    u := &User{}
    if err := db.Get(u, userID); err != nil {
        return nil, err
    }
    return u, nil
}
```

**Правильно:**
```go
func getUser(ctx context.Context, userID int) (*User, error) {
    u := &User{}
    if err := db.Get(u, userID, ctx); err != nil {
        return nil, fmt.Errorf("context: %w", err)
    }
    return u, nil
}
```

Таким образом, ты сразу оборачиваешь ошибку и указываешь контекст.

вы> Покажи, как правильно передать ctx в запрос к ClickHouse.


ассистент> Конечно, вот пример правильного использования `context.Context` при запросе к ClickHouse:

**Неправильно:**
```go
func getClickHouseData(query string) ([][]string, error) {
    rows, err := db.Query(query)
    if err != nil {
        return nil, err
    }
    defer rows.Close()
    return rows.Columns()
}
```

**Правильно:**
```go
func getClickHouseData(ctx context.Context, query string) ([][]string, error) {
    rows, err := db.Query(query, ctx)
    if err != nil {
        return nil, fmt.Errorf("context: %w", err)
    }
    defer rows.Close()
    return rows.Columns()
}
```

Таким образом, контекст передается в запрос к базе данных и обрабатывается в случае ошибки.

вы> /exit


In [9]:
log = fa.LogCell("Служебный лог — профиль olga")

**Служебный лог — профиль olga**

- `11:34:07` в `memory.json` нет профиля `olga` — память пустая
- `11:34:07` сеанс начат: user_id=`olga`, pid процесса `81`
- `11:34:07` не хватает фактов: name, go_level, go_experience, stack, project, weak_spots, goal
- `11:34:07` модель извлекла: `{"name": "Ольга", "go_level": "junior", "go_experience": "полгода"}`
- `11:34:07` записано в `memory.json`: name, go_level, go_experience
- `11:34:07` дальше: спросить stack
- `11:34:08` модель извлекла: `{"stack": "MySQL, gRPC"}`
- `11:34:08` записано в `memory.json`: stack
- `11:34:08` дальше: спросить project
- `11:34:08` сеанс завершён командой выхода

In [10]:
olga = fa.FactAgent("olga", MEMORY_FILE, log=log)
olga.chat(script=[
    'Привет, я Ольга, джун, пишу на Go полгода.',
    'MySQL и немного gRPC.',
    '/exit',
])

ассистент> Привет! Я код-стайлист по Go. Как тебя зовут?

вы> Привет, я Ольга, джун, пишу на Go полгода.


ассистент> С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?

вы> MySQL и немного gRPC.


ассистент> Над каким проектом ты сейчас работаешь?

вы> /exit


In [11]:
log = fa.LogCell("Служебный лог — guest возвращается")

**Служебный лог — guest возвращается**

- `11:34:08` прочитан `memory.json`, профиль `guest`: `{"name": "Мария", "go_level": "senior", "go_experience": "6 лет"}`
- `11:34:08` сеанс начат: user_id=`guest`, pid процесса `81`
- `11:34:08` не хватает фактов: stack, project, weak_spots, goal
- `11:34:09` модель извлекла: `{"stack": "PostgreSQL, Kafka, Linux"}`
- `11:34:09` записано в `memory.json`: stack
- `11:34:09` дальше: спросить project
- `11:34:10` модель извлекла: `{"project": "Платёжный шлюз для интернет-магазинов"}`
- `11:34:10` записано в `memory.json`: project
- `11:34:10` дальше: спросить weak_spots
- `11:34:10` модель извлекла: `{"weak_spots": "запись doc-комментарий"}`
- `11:34:10` записано в `memory.json`: weak_spots
- `11:34:10` дальше: спросить goal
- `11:34:11` модель извлекла: `{"goal": "чтобы вся команда писала в едином стиле"}`
- `11:34:11` записано в `memory.json`: goal
- `11:34:11` дальше: режим наставника — все факты собраны
- `11:34:12` сеанс завершён командой выхода

In [12]:
guest = fa.FactAgent("guest", MEMORY_FILE, log=log)
guest.chat(script=[
    'PostgreSQL, Kafka, Linux, пишу в GoLand.',
    'Платёжный шлюз для интернет-магазинов.',
    'Забываю писать doc-комментарии к экспортируемым функциям.',
    'Хочу, чтобы вся команда писала в едином стиле.',
    '/exit',
])

print(Path(MEMORY_FILE).read_text(encoding="utf-8"))

ассистент> Привет, Мария! Уже знаешь, что ты senior и работала 6 лет на Go. «С какими технологиями ты работаешь кроме Go: базы, инструменты, ОС?»

вы> PostgreSQL, Kafka, Linux, пишу в GoLand.


ассистент> Над каким проектом ты сейчас работаешь?

вы> Платёжный шлюз для интернет-магазинов.


ассистент> Что тебе сейчас даётся труднее всего в Go или в код-стайле?

вы> Забываю писать doc-комментарии к экспортируемым функциям.


ассистент> Чего ты хочешь добиться?

вы> Хочу, чтобы вся команда писала в едином стиле.


ассистент> Пример: функция, которая выполняет запрос в базу данных. Плохо: `func GetUser(id int) (User, error) { ... }`. По правилам: `func GetUser(ctx context.Context, id int) (User, error) { ... }`.

вы> /exit


{
  "artur": {
    "facts": {
      "name": "Артур",
      "go_level": "junior",
      "go_experience": "более года",
      "stack": "PostgreSQL, Docker, NixOS",
      "project": "сервис заказов с REST API",
      "weak_spots": "управление ошибками в Go",
      "goal": "писать идиоматичный Go и проходить ревью с первого раза"
    },
    "updated_at": "2026-10-23T11:33:54"
  },
  "guest": {
    "facts": {
      "name": "Мария",
      "go_level": "senior",
      "go_experience": "6 лет",
      "stack": "PostgreSQL, Kafka, Linux",
      "project": "Платёжный шлюз для интернет-магазинов",
      "weak_spots": "запись doc-комментарий",
      "goal": "чтобы вся команда писала в едином стиле"
    },
    "updated_at": "2026-10-23T11:34:11"
  },
  "dmitry": {
    "facts": {
      "name": "Дмитрий",
      "go_level": "middle",
      "go_experience": "три года",
      "stack": "Kafka, Redis, ClickHouse, Kubernetes",
      "project": "Биллинг",
      "weak_spots": "context.Context",
      "goal": "

## Свой диалог

```python
agent = fa.FactAgent(USER_ID, MEMORY_FILE, log=fa.LogCell())
agent.chat()   # /exit — выход
```